# Learning SPSS Syntax From Scratch
## A Complete, Hands-On Course Using the UAC Foods Nigeria Customer Survey

This course teaches SPSS **syntax** — the command language behind the point-and-click menus — from an absolute beginning through to regression and reliability analysis. Every command is demonstrated on one running dataset, so you build a real analysis project as you learn, instead of jumping between disconnected examples.

## About the Dataset

The dataset, `UAC_Foods_Survey.csv` / `UAC_Foods_Survey.xlsx`, is a simulated customer-satisfaction survey of 250 Nigerian consumers of UAC Foods brands (Gala, Supreme Ice Cream, Swan Water, Mr Chips, and Grand Malt). It was built for teaching purposes, with realistic patterns deliberately included: income relates to education, satisfaction sub-ratings drive overall satisfaction, and satisfaction drives willingness to recommend. This means the statistical tests in this course return genuinely interesting, discussable results rather than pure noise.

### Data Dictionary

| Variable | Type | Description |
|---|---|---|
| RespondentID | Numeric | Unique respondent identifier (1–250) |
| Region | String | Lagos, Abuja, Port Harcourt, Ibadan, Kano |
| Gender | String | Male, Female |
| Age | Numeric | Age in years (6 missing values, coded blank) |
| Education | String | Secondary, OND/HND, Bachelor's Degree, Postgraduate |
| Product | String | Main UAC Foods product purchased |
| MonthlyIncome_NGN000 | Numeric | Monthly income in thousands of Naira (5 missing values) |
| PurchaseFrequency | Numeric | Purchases per month (1–20) |
| LoyaltyYears | Numeric | Years as a customer of the brand (0–15) |
| Taste | Numeric | Satisfaction with taste, 1 (very dissatisfied) – 5 (very satisfied) |
| Price | Numeric | Satisfaction with price, 1–5 |
| Packaging | Numeric | Satisfaction with packaging, 1–5 |
| Availability | Numeric | Satisfaction with availability in stores, 1–5 |
| OverallSatisfaction | Numeric | Overall satisfaction, 1–5 |
| MonthlySpend_NGN | Numeric | Estimated monthly spend on the product, in Naira |
| WouldRecommend | String | Yes / No |

Keep this table close by — every syntax example below refers back to these exact variable names.

---

---

### How to use this notebook

Every SPSS command in this course is shown in a fenced syntax block, exactly as you would type it
into an SPSS **Syntax Editor** window (SPSS syntax cannot run inside a Jupyter/Python kernel — there
is no SPSS engine here). Alongside the key chapters, you'll also find **live, runnable Python cells**
(using `pandas`, `scipy`, and `statsmodels`) that compute the *same* result on the *same* dataset.
Run those cells to see actual output while you read the matching SPSS syntax — a good way to build
real intuition for what each SPSS command is doing under the hood.

Run the cell below first — every Python demo cell in this notebook depends on it.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

df = pd.read_csv('UAC_Foods_Survey.csv')
print(df.shape)
df.head()

(250, 16)


,RespondentID,Region,Gender,Age,Education,Product,MonthlyIncome_NGN000,PurchaseFrequency,LoyaltyYears,Taste,Price,Packaging,Availability,OverallSatisfaction,MonthlySpend_NGN,WouldRecommend
0,1,Ibadan,Female,56.0,OND/HND,Grand Malt,72.0,8,10,3,2,4,4,3,1228.0,No
1,2,Abuja,Female,53.0,Bachelor's Degree,Supreme Ice Cream,140.0,9,12,3,3,4,5,3,1311.0,Yes
2,3,Kano,Male,44.0,Secondary,Supreme Ice Cream,84.0,3,5,3,3,4,3,4,786.0,Yes
3,4,Port Harcourt,Male,60.0,Bachelor's Degree,Swan Water,145.0,9,8,3,4,4,1,3,1536.0,Yes
4,5,Lagos,Male,24.0,Bachelor's Degree,Swan Water,120.0,8,0,3,2,4,4,3,1429.0,Yes


## Chapter 1: Getting Started with SPSS Syntax

### 1.1 Why learn syntax instead of just using the menus?

The point-and-click menus in SPSS are really just a friendly front end that *writes syntax for you* and then runs it. Learning syntax directly gives you three things the menus can't:

- **Reproducibility** — re-run an entire analysis, exactly, in seconds, whenever the data changes.
- **Documentation** — a syntax file is a readable record of every decision you made.
- **Power** — some options are only available in syntax, not in the dialog boxes.

### 1.2 The Syntax Window

Open it with **File > New > Syntax**. You'll write commands there and run them with:

- **Ctrl+R**, or
- highlight the command(s) and click the "Run" (play) icon, or
- **Run > All**

### 1.3 Anatomy of a command

Every SPSS syntax command follows the same shape:

```spss
COMMAND KEYWORD subcommand=value /subcommand2=value2.
```

Rules that trip up beginners:

1. **Every command ends with a period (`.`)** This is the single most common source of errors — a missing period causes SPSS to keep reading the next line as part of the same command.
2. Commands can span multiple lines freely; SPSS doesn't care about line breaks, only the final period.
3. Subcommands are introduced with a forward slash `/`.
4. SPSS syntax is **not case sensitive** for command names (`FREQUENCIES` = `frequencies`), but string *data values* inside quotes ARE case sensitive (`"Lagos"` ≠ `"lagos"`).
5. Comments start with an asterisk `*` and end with a period, or use `/* ... */` inline.

```spss
* This is a comment describing what the next command does.
FREQUENCIES VARIABLES=Region.

/* This is an inline comment */ FREQUENCIES VARIABLES=Gender.
```

### 1.4 A first command: checking SPSS is alive

```spss
GET FILE = 'C:\UAC_Foods\UAC_Foods_Survey.sav'.
DISPLAY DICTIONARY.
```

`DISPLAY DICTIONARY` prints every variable's name, type, and labels currently loaded — a useful sanity check after any import.

---

## Chapter 2: Importing the UAC Foods Data

You will almost never start from an existing `.sav` file in the real world — you'll import from Excel, CSV, or a text file. This chapter shows all three, using the two files provided with this course.

### 2.1 Importing the CSV file

```spss
GET DATA
  /TYPE=TXT
  /FILE='C:\UAC_Foods\UAC_Foods_Survey.csv'
  /DELIMITERS=","
  /QUALIFIER='"'
  /ARRANGEMENT=DELIMITED
  /FIRSTCASE=2
  /VARIABLES=
    RespondentID F8.0
    Region A20
    Gender A10
    Age F8.0
    Education A20
    Product A20
    MonthlyIncome_NGN000 F8.0
    PurchaseFrequency F8.0
    LoyaltyYears F8.0
    Taste F8.0
    Price F8.0
    Packaging F8.0
    Availability F8.0
    OverallSatisfaction F8.0
    MonthlySpend_NGN F8.0
    WouldRecommend A10.
CACHE.
EXECUTE.
```

Key points:

- `/FIRSTCASE=2` tells SPSS the real data starts on row 2, because row 1 holds the column headers.
- `A20` means an alphanumeric (string) field up to 20 characters; `F8.0` means a numeric field, 8 digits wide, 0 decimals.
- `CACHE.` and `EXECUTE.` force SPSS to actually read the file into memory immediately rather than waiting — good practice right after any `GET DATA`.

### 2.2 Importing the Excel file

Excel import is simpler because SPSS reads the column types automatically:

```spss
GET DATA
  /TYPE=XLSX
  /FILE='C:\UAC_Foods\UAC_Foods_Survey.xlsx'
  /SHEET=NAME 'UAC_Foods_Survey'
  /CELLRANGE=FULL
  /READNAMES=ON
  /DATATYPEMIN PERCENTAGE=95.0.
EXECUTE.
```

- `/READNAMES=ON` uses row 1 as variable names.
- `/CELLRANGE=FULL` reads every populated cell; you could instead specify e.g. `/CELLRANGE=RANGE 'A1:P251'`.

### 2.3 Saving as a native `.sav` file

Once imported, save a native SPSS file so future sessions start instantly, without re-importing:

```spss
SAVE OUTFILE='C:\UAC_Foods\UAC_Foods_Survey.sav'.
```

### 2.4 Quick structural check after any import

```spss
DISPLAY DICTIONARY.
DESCRIPTIVES VARIABLES=Age MonthlyIncome_NGN000 PurchaseFrequency.
FREQUENCIES VARIABLES=Region Gender Product.
```

Always run this trio after import. It confirms variable types are correct, no column is shifted, and missing values look plausible.

---

### Try it yourself — Python check for Chapter 2

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [2]:
# Python equivalent of Chapter 2: importing the data
df_csv = pd.read_csv('UAC_Foods_Survey.csv')
df_xlsx = pd.read_excel('UAC_Foods_Survey.xlsx', sheet_name='UAC_Foods_Survey')

print("CSV shape: ", df_csv.shape)
print("Excel shape:", df_xlsx.shape)
df_csv.dtypes

CSV shape:  (250, 16)
Excel shape: (250, 16)


RespondentID              int64
Region                      str
Gender                      str
Age                     float64
Education                   str
Product                     str
MonthlyIncome_NGN000    float64
PurchaseFrequency         int64
LoyaltyYears              int64
Taste                     int64
Price                     int64
Packaging                 int64
Availability              int64
OverallSatisfaction       int64
MonthlySpend_NGN        float64
WouldRecommend              str
dtype: object

## Chapter 3: Defining Your Variables Properly

Raw imported data works, but it's not yet *documented* data. This chapter turns bare column names into a self-explaining dataset — essential before you hand syntax to a colleague, a supervisor, or your future self.

### 3.1 VARIABLE LABELS — full-text descriptions

```spss
VARIABLE LABELS
  RespondentID 'Unique Respondent Identifier'
  Region 'Nigerian Region of Residence'
  Gender 'Respondent Gender'
  Age 'Respondent Age in Years'
  Education 'Highest Level of Education Completed'
  Product 'Main UAC Foods Product Purchased'
  MonthlyIncome_NGN000 'Monthly Income (Thousands of Naira)'
  PurchaseFrequency 'Number of Purchases per Month'
  LoyaltyYears 'Years as a Customer of the Brand'
  Taste 'Satisfaction with Taste'
  Price 'Satisfaction with Price'
  Packaging 'Satisfaction with Packaging'
  Availability 'Satisfaction with In-Store Availability'
  OverallSatisfaction 'Overall Satisfaction with the Product'
  MonthlySpend_NGN 'Estimated Monthly Spend (Naira)'
  WouldRecommend 'Would Recommend This Product to Others'.
```

Now every table and chart shows these descriptive labels instead of the raw variable names.

### 3.2 VALUE LABELS — labeling coded categories

Our satisfaction items are stored as numbers 1–5. VALUE LABELS attaches meaning to each number:

```spss
VALUE LABELS Taste Price Packaging Availability OverallSatisfaction
  1 'Very Dissatisfied'
  2 'Dissatisfied'
  3 'Neutral'
  4 'Satisfied'
  5 'Very Satisfied'.

VALUE LABELS WouldRecommend
  'Yes' 'Would Recommend'
  'No'  'Would Not Recommend'.
```

Note the difference: numeric variables (Taste, Price...) take *unquoted* numbers as values; string variables (WouldRecommend) take *quoted* strings as values.

### 3.3 VARIABLE LEVEL — declaring measurement level

SPSS statistics and charts behave differently depending on whether a variable is Nominal, Ordinal, or Scale. Declare this explicitly rather than trusting the auto-detected default:

```spss
VARIABLE LEVEL
  RespondentID (NOMINAL)
  Region Gender Education Product WouldRecommend (NOMINAL)
  Taste Price Packaging Availability OverallSatisfaction (ORDINAL)
  Age MonthlyIncome_NGN000 PurchaseFrequency LoyaltyYears MonthlySpend_NGN (SCALE).
```

### 3.4 MISSING VALUES — flagging codes that don't represent real data

Our Age and MonthlyIncome_NGN000 columns already have blank system-missing cells from the import. But suppose, in a future wave of this survey, a "999" code is used for "declined to answer". You'd declare it like this:

```spss
MISSING VALUES Age (999) MonthlyIncome_NGN000 (999).
```

For our current file, blanks already imported as **system-missing** (shown as a period `.` in Data View), so no MISSING VALUES command is required — but it's important to know the command exists for the day a numeric "missing code" shows up instead of a blank.

### 3.5 VARIABLE WIDTH and FORMATS (cosmetic, but professional)

```spss
FORMATS MonthlyIncome_NGN000 MonthlySpend_NGN (F8.0).
VARIABLE WIDTH Region Product (10).
```

---

## Chapter 4: Data Management I — Creating and Transforming Variables

### 4.1 COMPUTE — creating a new variable from a formula

Let's create a total satisfaction *index* by averaging the four satisfaction sub-items:

```spss
COMPUTE SatisfactionIndex = MEAN(Taste, Price, Packaging, Availability).
VARIABLE LABELS SatisfactionIndex 'Average Satisfaction Index (Taste/Price/Packaging/Availability)'.
FORMATS SatisfactionIndex (F5.2).
EXECUTE.
```

Also useful: create an **annual spend** figure from the monthly one.

```spss
COMPUTE AnnualSpend_NGN = MonthlySpend_NGN * 12.
VARIABLE LABELS AnnualSpend_NGN 'Estimated Annual Spend (Naira)'.
EXECUTE.
```

`EXECUTE.` forces SPSS to actually pass through the data and apply the transformation immediately (transformations otherwise wait until the next command that reads the data).

### 4.2 RECODE — collapsing categories

**RECODE INTO a new variable** (always preferred over recoding in place, so you keep the original):

```spss
RECODE Age (18 THRU 29 = 1) (30 THRU 44 = 2) (45 THRU 59 = 3) (60 THRU HIGHEST = 4) (MISSING=SYSMIS)
  INTO AgeGroup.
VARIABLE LABELS AgeGroup 'Age Group (Banded)'.
VALUE LABELS AgeGroup 1 'Under 30' 2 '30-44' 3 '45-59' 4 '60 and above'.
EXECUTE.
```

Banding income the same way:

```spss
RECODE MonthlyIncome_NGN000 (LOWEST THRU 79.99 = 1) (80 THRU 149.99 = 2) (150 THRU HIGHEST = 3) (MISSING=SYSMIS)
  INTO IncomeBand.
VARIABLE LABELS IncomeBand 'Monthly Income Band'.
VALUE LABELS IncomeBand 1 'Low (< N80,000)' 2 'Middle (N80,000-149,999)' 3 'High (N150,000+)'.
EXECUTE.
```

`THRU` defines an inclusive range; `LOWEST` and `HIGHEST` avoid hard-coding the true minimum/maximum; `MISSING=SYSMIS` carries missing values through instead of accidentally recoding them into a real category — a very common beginner mistake is to let system-missing values silently fall into the first range.

### 4.3 IF — conditional assignment

`IF` assigns a value to an *existing* variable only when a condition is true:

```spss
COMPUTE HighValueCustomer = 0.
IF (MonthlySpend_NGN > 2000 & LoyaltyYears >= 5) HighValueCustomer = 1.
VARIABLE LABELS HighValueCustomer 'High-Value Loyal Customer Flag'.
VALUE LABELS HighValueCustomer 0 'No' 1 'Yes'.
EXECUTE.
```

Note the pattern: initialize the variable first with `COMPUTE`, then narrow it with one or more `IF` statements.

### 4.4 DO IF / ELSE IF / END IF — multi-branch logic

For more than two branches, `DO IF` is clearer than stacking `IF` statements:

```spss
DO IF (OverallSatisfaction >= 4).
  COMPUTE SatisfactionTier = 3.
ELSE IF (OverallSatisfaction = 3).
  COMPUTE SatisfactionTier = 2.
ELSE.
  COMPUTE SatisfactionTier = 1.
END IF.
VARIABLE LABELS SatisfactionTier 'Satisfaction Tier'.
VALUE LABELS SatisfactionTier 1 'Low' 2 'Medium' 3 'High'.
EXECUTE.
```

### 4.5 COUNT — counting occurrences across variables

Suppose we want to know, for each respondent, how many of the four satisfaction items scored the maximum (5):

```spss
COUNT MaxScoreCount = Taste Price Packaging Availability (5).
VARIABLE LABELS MaxScoreCount 'Number of Satisfaction Items Rated 5 (Very Satisfied)'.
EXECUTE.
```

### 4.6 STRING and numeric-to-string conversions

```spss
STRING RegionCode (A2).
IF (Region = 'Lagos') RegionCode = 'LA'.
IF (Region = 'Abuja') RegionCode = 'AB'.
IF (Region = 'Port Harcourt') RegionCode = 'PH'.
IF (Region = 'Ibadan') RegionCode = 'IB'.
IF (Region = 'Kano') RegionCode = 'KN'.
EXECUTE.
```

---

### Try it yourself — Python check for Chapter 4

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [3]:
# Python equivalent of Chapter 4: COMPUTE and RECODE
df['SatisfactionIndex'] = df[['Taste', 'Price', 'Packaging', 'Availability']].mean(axis=1)

bins = [17, 29, 44, 59, 200]
labels = ['Under 30', '30-44', '45-59', '60 and above']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

df[['Taste', 'Price', 'Packaging', 'Availability', 'SatisfactionIndex', 'Age', 'AgeGroup']].head(10)

,Taste,Price,Packaging,Availability,SatisfactionIndex,Age,AgeGroup
0,3,2,4,4,3.25,56.0,45-59
1,3,3,4,5,3.75,53.0,45-59
2,3,3,4,3,3.25,44.0,30-44
3,3,4,4,1,3.00,60.0,60 and above
4,3,2,4,4,3.25,24.0,Under 30
5,4,2,4,4,3.50,62.0,60 and above
6,4,1,4,3,3.00,58.0,45-59
7,4,3,4,3,3.50,42.0,30-44
8,4,2,3,3,3.00,53.0,45-59
9,5,3,2,5,3.75,42.0,30-44


## Chapter 5: Data Management II — Selecting, Sorting, and Aggregating Cases

### 5.1 SORT CASES

```spss
SORT CASES BY Region (A) OverallSatisfaction (D).
```

`(A)` = ascending, `(D)` = descending. Sorting is often a prerequisite for other commands (like `AGGREGATE` with the `BREAK` subcommand relying on presorted groups in older syntax styles, though modern `AGGREGATE` sorts internally too).

### 5.2 SELECT IF — permanently keeping only certain cases

```spss
* Keep only Lagos respondents for a Lagos-specific report.
SELECT IF (Region = 'Lagos').
EXECUTE.
```

Use this with care — it permanently deletes non-matching cases from the working data file for the remainder of the session (or until you re-open the file). For a temporary subset, use `TEMPORARY` (Section 5.3) or `FILTER` instead.

### 5.3 TEMPORARY — a one-command-only restriction

```spss
TEMPORARY.
SELECT IF (Product = 'Gala').
FREQUENCIES VARIABLES=OverallSatisfaction.
```

Because of `TEMPORARY`, the `SELECT IF` above applies **only** to the very next command (the `FREQUENCIES`) and the full dataset is restored immediately afterward.

### 5.4 FILTER — a reversible, on/off subset

```spss
COMPUTE FilterHighIncome = (IncomeBand = 3).
FILTER BY FilterHighIncome.
* Any analysis run now only includes high-income respondents...
FREQUENCIES VARIABLES=Product.
* ...turn the filter back off when done:
FILTER OFF.
USE ALL.
EXECUTE.
```

### 5.5 AGGREGATE — building a summary table as a new dataset

```spss
AGGREGATE
  /OUTFILE='C:\UAC_Foods\Region_Summary.sav'
  /BREAK=Region
  /MeanSatisfaction = MEAN(OverallSatisfaction)
  /MeanSpend = MEAN(MonthlySpend_NGN)
  /N_Respondents = N.
```

This creates a brand-new file with one row per Region, containing the average satisfaction, average spend, and respondent count — perfect input for a regional comparison chart.

### 5.6 WEIGHT — applying survey weights

If Kano and Port Harcourt were under-sampled relative to their real population share, you might weight cases to correct this:

```spss
COMPUTE SurveyWeight = 1.
IF (Region = 'Kano') SurveyWeight = 1.3.
IF (Region = 'Port Harcourt') SurveyWeight = 1.2.
WEIGHT BY SurveyWeight.
EXECUTE.
* Turn weighting off again with:
WEIGHT OFF.
```

---

## Chapter 6: Descriptive Statistics

### 6.1 FREQUENCIES — categorical and ordinal summaries

```spss
FREQUENCIES VARIABLES=Region Gender Education Product WouldRecommend
  /ORDER=ANALYSIS.
```

Add a bar chart directly from the command:

```spss
FREQUENCIES VARIABLES=Product
  /BARCHART PERCENT
  /ORDER=ANALYSIS.
```

### 6.2 DESCRIPTIVES — scale-variable summaries (mean, SD, min, max)

```spss
DESCRIPTIVES VARIABLES=Age MonthlyIncome_NGN000 PurchaseFrequency LoyaltyYears MonthlySpend_NGN
  /STATISTICS=MEAN STDDEV MIN MAX RANGE SEMEAN
  /SORT=MEAN (A).
```

`/SORT` orders the output table by a chosen statistic — handy in a report with many variables.

To also standardize a variable (create a z-score version), add `/SAVE`:

```spss
DESCRIPTIVES VARIABLES=MonthlySpend_NGN /SAVE.
```

This creates a new variable `ZMonthlySpend_NGN` automatically.

### 6.3 EXAMINE — deeper distributional diagnostics

`EXAMINE` gives you skewness, kurtosis, a Shapiro-Wilk normality test, and boxplots — essential before choosing parametric vs. non-parametric tests later.

```spss
EXAMINE VARIABLES=MonthlySpend_NGN BY Gender
  /PLOT BOXPLOT HISTOGRAM
  /STATISTICS DESCRIPTIVES
  /CINTERVAL 95
  /MISSING LISTWISE.
```

### 6.4 Frequencies with percentiles for ordinal items

```spss
FREQUENCIES VARIABLES=OverallSatisfaction
  /FORMAT=NOTABLE
  /PERCENTILES=25 50 75
  /STATISTICS=MEDIAN MODE.
```

---

### Try it yourself — Python check for Chapter 6

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [4]:
# Python equivalent of Chapter 6: FREQUENCIES and DESCRIPTIVES
print(df['Product'].value_counts())
print()
df[['Age', 'MonthlyIncome_NGN000', 'PurchaseFrequency', 'LoyaltyYears', 'MonthlySpend_NGN']].describe()

Product
Gala                 75
Supreme Ice Cream    57
Swan Water           45
Grand Malt           40
Mr Chips             33
Name: count, dtype: int64



,Age,MonthlyIncome_NGN000,PurchaseFrequency,LoyaltyYears,MonthlySpend_NGN
count,244.000000,245.000000,250.000000,250.000000,250.000000
mean,42.807377,120.191837,10.812000,7.404000,1651.884000
std,13.873613,52.275907,5.836516,4.481195,894.937196
min,18.000000,30.000000,1.000000,0.000000,200.000000
25%,32.000000,80.000000,6.000000,3.000000,902.250000
50%,42.500000,119.000000,11.000000,7.000000,1666.500000
75%,54.250000,156.000000,16.000000,11.000000,2405.250000
max,65.000000,253.000000,20.000000,15.000000,3564.000000


## Chapter 7: Cross-Tabulation and the Chi-Square Test

Cross-tabulation is the workhorse of survey analysis: does one categorical variable relate to another?

### 7.1 A basic two-way table

```spss
CROSSTABS
  /TABLES=Gender BY WouldRecommend
  /FORMAT=AVALUE TABLES
  /CELLS=COUNT ROW COLUMN TOTAL
  /COUNT ROUND CELL.
```

`/CELLS=COUNT ROW COLUMN TOTAL` prints raw counts plus row%, column%, and total% — the combination you'll want in almost every real report.

### 7.2 Adding the Chi-Square test of independence

```spss
CROSSTABS
  /TABLES=Region BY WouldRecommend
  /FORMAT=AVALUE TABLES
  /STATISTICS=CHISQ PHI
  /CELLS=COUNT ROW
  /COUNT ROUND CELL.
```

- `CHISQ` requests the Pearson Chi-Square statistic and its p-value (Asymp. Sig.).
- `PHI` requests the Phi and Cramér's V effect-size measures, useful when Chi-Square is significant and you want to know *how strongly*, not just *whether*, the variables are associated.

**Reading the output:** if "Asymp. Sig. (2-sided)" is below .05, conclude Region and WouldRecommend are not independent — recommendation rates genuinely differ by region in this sample.

### 7.3 Three-way (layered) tables

```spss
CROSSTABS
  /TABLES=Product BY WouldRecommend BY Gender
  /STATISTICS=CHISQ
  /CELLS=COUNT ROW.
```

This produces a separate Product × WouldRecommend table for each Gender, letting you check whether an association holds for both men and women separately.

---

### Try it yourself — Python check for Chapter 7

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [5]:
# Python equivalent of Chapter 7: CROSSTABS with Chi-Square
ct = pd.crosstab(df['Region'], df['WouldRecommend'])
print(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"\nChi-square = {chi2:.3f}, df = {dof}, p-value = {p:.4f}")

WouldRecommend  No  Yes
Region                 
Abuja           12   35
Ibadan          14   27
Kano             2   30
Lagos           18   69
Port Harcourt    8   35

Chi-square = 8.963, df = 4, p-value = 0.0620


## Chapter 8: Comparing Means — T-Tests and ANOVA

### 8.1 Independent-Samples T-Test

Does average monthly spend differ between men and women?

```spss
T-TEST GROUPS=Gender('Male' 'Female')
  /VARIABLES=MonthlySpend_NGN
  /CRITERIA=CI(.95)
  /MISSING=ANALYSIS.
```

**Reading the output:** first check **Levene's Test for Equality of Variances**. If its Sig. is above .05, read the "Equal variances assumed" row for the t, df, and Sig. (2-tailed); if below .05, read the "Equal variances not assumed" row instead.

### 8.2 Paired-Samples T-Test

Suppose we wanted to compare Taste satisfaction against Packaging satisfaction *within the same respondents* (a meaningful paired comparison since both are 1–5 scales from the same people):

```spss
T-TEST PAIRS=Taste WITH Packaging (PAIRED)
  /CRITERIA=CI(.95)
  /MISSING=ANALYSIS.
```

### 8.3 One-Sample T-Test

Testing whether average OverallSatisfaction differs from the survey's target benchmark of 3.5:

```spss
T-TEST /TESTVAL=3.5
  /VARIABLES=OverallSatisfaction
  /CRITERIA=CI(.95)
  /MISSING=ANALYSIS.
```

### 8.4 One-Way ANOVA — comparing more than two groups

Does average MonthlySpend_NGN differ across the five Regions?

```spss
ONEWAY MonthlySpend_NGN BY Region
  /STATISTICS DESCRIPTIVES HOMOGENEITY
  /MISSING ANALYSIS
  /POSTHOC=TUKEY BONFERRONI ALPHA(0.05).
```

- `HOMOGENEITY` requests Levene's test — check this before trusting the standard ANOVA F-test.
- `/POSTHOC=TUKEY BONFERRONI` tells you *which specific pairs* of regions differ, once the overall F-test is significant — the overall ANOVA only tells you "somewhere a difference exists", not where.

### 8.5 Two-way ANOVA with UNIANOVA (bonus, for when one factor isn't enough)

```spss
UNIANOVA MonthlySpend_NGN BY Region Gender
  /METHOD=SSTYPE(3)
  /INTERCEPT=INCLUDE
  /PRINT=DESCRIPTIVE ETASQ
  /CRITERIA=ALPHA(0.05)
  /DESIGN=Region Gender Region*Gender.
```

The `Region*Gender` term tests whether the effect of Region on spend depends on Gender (an interaction effect).

---

### Try it yourself — Python check for Chapter 8

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [6]:
# Python equivalent of Chapter 8: independent t-test and one-way ANOVA
male = df.loc[df['Gender'] == 'Male', 'MonthlySpend_NGN']
female = df.loc[df['Gender'] == 'Female', 'MonthlySpend_NGN']
t_stat, p_val = stats.ttest_ind(male, female, equal_var=True)
print(f"Independent t-test (Male vs Female spend): t = {t_stat:.3f}, p = {p_val:.4f}")

groups = [g['MonthlySpend_NGN'].values for _, g in df.groupby('Region')]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"One-way ANOVA (spend by Region): F = {f_stat:.3f}, p = {p_anova:.4f}")

Independent t-test (Male vs Female spend): t = 0.519, p = 0.6042
One-way ANOVA (spend by Region): F = 0.424, p = 0.7913


## Chapter 9: Correlation

### 9.1 Pearson correlation matrix

```spss
CORRELATIONS
  /VARIABLES=Taste Price Packaging Availability OverallSatisfaction MonthlySpend_NGN
  /PRINT=TWOTAIL NOSIG
  /MISSING=PAIRWISE.
```

`/PRINT=TWOTAIL NOSIG` requests a two-tailed test and flags non-significant correlations, so significant ones stand out.

### 9.2 Spearman correlation (for ordinal / non-normal data)

Since our satisfaction items are ordinal Likert scales, a Spearman rank correlation is often more defensible than Pearson:

```spss
NONPAR CORR
  /VARIABLES=Taste Price Packaging Availability OverallSatisfaction
  /PRINT=SPEARMAN TWOTAIL
  /MISSING=PAIRWISE.
```

---

### Try it yourself — Python check for Chapter 9

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [7]:
# Python equivalent of Chapter 9: Pearson correlation matrix
corr_vars = ['Taste', 'Price', 'Packaging', 'Availability', 'OverallSatisfaction', 'MonthlySpend_NGN']
df[corr_vars].corr(method='pearson').round(2)

,Taste,Price,Packaging,Availability,OverallSatisfaction,MonthlySpend_NGN
Taste,1.00,-0.02,-0.05,0.05,0.35,0.04
Price,-0.02,1.00,-0.00,-0.04,0.35,0.12
Packaging,-0.05,-0.00,1.00,0.04,0.22,-0.02
Availability,0.05,-0.04,0.04,1.00,0.39,-0.03
OverallSatisfaction,0.35,0.35,0.22,0.39,1.00,0.02
MonthlySpend_NGN,0.04,0.12,-0.02,-0.03,0.02,1.00


## Chapter 10: Reliability Analysis (Cronbach's Alpha)

Before treating Taste, Price, Packaging, and Availability as one combined "satisfaction index" (as we did in Chapter 4), we should check that they reliably measure a single underlying construct:

```spss
RELIABILITY
  /VARIABLES=Taste Price Packaging Availability
  /SCALE('Satisfaction Scale') ALL
  /MODEL=ALPHA
  /STATISTICS=DESCRIPTIVE SCALE CORR
  /SUMMARY=TOTAL.
```

**Rule of thumb for Cronbach's Alpha:** ≥ .70 is generally considered acceptable internal consistency for a scale used in applied research; ≥ .80 is good. The `/SUMMARY=TOTAL` subcommand also prints "Alpha if Item Deleted" for each item — useful for spotting a weak item that's dragging reliability down.

---

### Try it yourself — Python check for Chapter 10

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [8]:
# Python equivalent of Chapter 10: Cronbach's Alpha for the satisfaction scale
def cronbach_alpha(items_df):
    items_df = items_df.dropna()
    item_vars = items_df.var(axis=0, ddof=1)
    total_var = items_df.sum(axis=1).var(ddof=1)
    k = items_df.shape[1]
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

alpha = cronbach_alpha(df[['Taste', 'Price', 'Packaging', 'Availability']])
print(f"Cronbach's Alpha = {alpha:.3f}")

Cronbach's Alpha = -0.016


## Chapter 11: Regression Analysis

### 11.1 Simple linear regression

Does purchase frequency predict monthly spend?

```spss
REGRESSION
  /MISSING LISTWISE
  /STATISTICS COEFF OUTS R ANOVA
  /DEPENDENT MonthlySpend_NGN
  /METHOD=ENTER PurchaseFrequency.
```

### 11.2 Multiple linear regression

Now with several predictors together:

```spss
REGRESSION
  /MISSING LISTWISE
  /STATISTICS COEFF OUTS R ANOVA COLLIN TOL
  /DEPENDENT MonthlySpend_NGN
  /METHOD=ENTER MonthlyIncome_NGN000 PurchaseFrequency LoyaltyYears OverallSatisfaction
  /SCATTERPLOT=(*ZRESID,*ZPRED)
  /SAVE PRED RESID.
```

- `COLLIN TOL` requests collinearity diagnostics (VIF, Tolerance) — check that no predictor has Tolerance below ~0.10 (a sign of problematic multicollinearity).
- `/SCATTERPLOT=(*ZRESID,*ZPRED)` produces the standard residuals-vs-predicted plot for checking homoscedasticity.
- `/SAVE PRED RESID` writes the model's predicted values and residuals back into the dataset as new columns.

**Reading the output:** the Model Summary table's R² tells you the proportion of variance in MonthlySpend_NGN explained jointly by all predictors; the Coefficients table's Sig. column tells you which individual predictors are statistically significant while holding the others constant.

### 11.3 Stepwise / hierarchical entry (bonus)

```spss
REGRESSION
  /MISSING LISTWISE
  /STATISTICS COEFF OUTS R ANOVA CHANGE
  /DEPENDENT MonthlySpend_NGN
  /METHOD=ENTER MonthlyIncome_NGN000
  /METHOD=ENTER PurchaseFrequency LoyaltyYears OverallSatisfaction.
```

Two `/METHOD=ENTER` blocks create a **hierarchical** regression: Block 1 enters income alone; Block 2 adds the remaining predictors. `CHANGE` in the STATISTICS list reports the R² Change between blocks — did the added predictors explain significantly more variance?

---

### Try it yourself — Python check for Chapter 11

Run the cell below to compute the same result directly on `UAC_Foods_Survey.csv` using pandas/scipy/statsmodels.

In [9]:
# Python equivalent of Chapter 11: multiple linear regression
import statsmodels.api as sm

reg_df = df.dropna(subset=['MonthlyIncome_NGN000', 'PurchaseFrequency', 'LoyaltyYears',
                            'OverallSatisfaction', 'MonthlySpend_NGN'])
X = reg_df[['MonthlyIncome_NGN000', 'PurchaseFrequency', 'LoyaltyYears', 'OverallSatisfaction']]
X = sm.add_constant(X)
y = reg_df['MonthlySpend_NGN']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       MonthlySpend_NGN   R-squared:                       0.892
Model:                            OLS   Adj. R-squared:                  0.890
Method:                 Least Squares   F-statistic:                     493.2
Date:                Thu, 16 Jul 2026   Prob (F-statistic):          1.82e-114
Time:                        21:23:17   Log-Likelihood:                -1740.5
No. Observations:                 245   AIC:                             3491.
Df Residuals:                     240   BIC:                             3508.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  117.2399 

## Chapter 12: Non-Parametric Tests (When Assumptions Don't Hold)

### 12.1 Mann-Whitney U (non-parametric alternative to independent t-test)

```spss
NPAR TESTS
  /M-W= OverallSatisfaction BY Gender(1 2)
  /MISSING ANALYSIS.
```

Note: Mann-Whitney in this syntax form expects numeric group codes. Since Gender is a string here, use the dialog-generated form instead, which SPSS auto-writes as:

```spss
NPTESTS
  /INDEPENDENT TEST (OverallSatisfaction) GROUP (Gender)
  /MISSING SCOPE=ANALYSIS USERMISSING=EXCLUDE
  /CRITERIA ALPHA=0.05 CILEVEL=95.
```

### 12.2 Kruskal-Wallis (non-parametric alternative to one-way ANOVA)

```spss
NPTESTS
  /INDEPENDENT TEST (MonthlySpend_NGN) GROUP (Region)
  /MISSING SCOPE=ANALYSIS USERMISSING=EXCLUDE
  /CRITERIA ALPHA=0.05 CILEVEL=95.
```

### 12.3 Chi-Square goodness-of-fit test

Testing whether the five products were purchased in *equal* proportions (rather than testing association *between* two variables, as CROSSTABS does):

```spss
NPAR TESTS
  /CHISQUARE=Product
  /EXPECTED=EQUAL
  /MISSING ANALYSIS.
```

---

## Chapter 13: Basic Charts from Syntax

### 13.1 Bar chart of a categorical variable

```spss
GRAPH
  /BAR(SIMPLE)=COUNT BY Product
  /MISSING=REPORT.
```

### 13.2 Clustered bar chart (two variables at once)

```spss
GRAPH
  /BAR(GROUPED)=COUNT BY Region BY WouldRecommend
  /MISSING=REPORT.
```

### 13.3 Scatterplot with a fit line

```spss
GRAPH
  /SCATTERPLOT(BIVAR)=MonthlyIncome_NGN000 WITH MonthlySpend_NGN
  /MISSING=LISTWISE.
```

### 13.4 Histogram with a normal curve overlay

```spss
FREQUENCIES VARIABLES=MonthlySpend_NGN
  /FORMAT=NOTABLE
  /HISTOGRAM NORMAL.
```

---

## Chapter 14: Output and File Management

### 14.1 Routing output to a Word-readable file

```spss
OUTPUT SAVE OUTFILE='C:\UAC_Foods\UAC_Foods_Results.spv'.
```

### 14.2 Exporting output directly to Word or PDF

```spss
OUTPUT EXPORT
  /CONTENTS  EXPORT=ALL
  /DOCUMENTFILE  DOCUMENTFILE='C:\UAC_Foods\UAC_Foods_Results.docx'
  /DOCUMENTTYPE  DOCUMENTTYPE=WORD.
```

### 14.3 Exporting the working data back out to CSV or Excel

```spss
SAVE TRANSLATE OUTFILE='C:\UAC_Foods\UAC_Foods_Clean.csv'
  /TYPE=CSV
  /FIELDNAMES
  /REPLACE.

SAVE TRANSLATE OUTFILE='C:\UAC_Foods\UAC_Foods_Clean.xlsx'
  /TYPE=XLSX
  /VERSION=12
  /FIELDNAMES
  /REPLACE.
```

### 14.4 A tip for learning faster: let SPSS write syntax for you

Even as you master syntax, you can accelerate learning by running any dialog box once through the menus, then clicking **Paste** instead of **OK**. SPSS writes the exact syntax for what you just configured into a new Syntax window — a fast way to discover subcommands you haven't memorized yet.

---

## Chapter 15: Putting It All Together — A Capstone Analysis Script

This single script chains everything above into one coherent, ready-to-run project analyzing what drives satisfaction and recommendation of UAC Foods products.

```spss
* ============================================================
* CAPSTONE SCRIPT: UAC Foods Customer Satisfaction Analysis
* ============================================================.

* 1. Import.
GET DATA
  /TYPE=XLSX
  /FILE='C:\UAC_Foods\UAC_Foods_Survey.xlsx'
  /SHEET=NAME 'UAC_Foods_Survey'
  /CELLRANGE=FULL
  /READNAMES=ON.
EXECUTE.

* 2. Label variables and values.
VARIABLE LABELS
  OverallSatisfaction 'Overall Satisfaction with the Product'
  MonthlySpend_NGN 'Estimated Monthly Spend (Naira)'.
VALUE LABELS Taste Price Packaging Availability OverallSatisfaction
  1 'Very Dissatisfied' 2 'Dissatisfied' 3 'Neutral' 4 'Satisfied' 5 'Very Satisfied'.

* 3. Build derived variables.
COMPUTE SatisfactionIndex = MEAN(Taste, Price, Packaging, Availability).
RECODE Age (18 THRU 29=1)(30 THRU 44=2)(45 THRU 59=3)(60 THRU HIGHEST=4)(MISSING=SYSMIS) INTO AgeGroup.
VALUE LABELS AgeGroup 1 'Under 30' 2 '30-44' 3 '45-59' 4 '60+'.
EXECUTE.

* 4. Describe the sample.
FREQUENCIES VARIABLES=Region Gender Product WouldRecommend.
DESCRIPTIVES VARIABLES=Age MonthlyIncome_NGN000 MonthlySpend_NGN /STATISTICS=MEAN STDDEV MIN MAX.

* 5. Check scale reliability before using SatisfactionIndex further.
RELIABILITY /VARIABLES=Taste Price Packaging Availability /MODEL=ALPHA.

* 6. Test group differences.
T-TEST GROUPS=Gender('Male' 'Female') /VARIABLES=MonthlySpend_NGN.
ONEWAY MonthlySpend_NGN BY Region /STATISTICS DESCRIPTIVES HOMOGENEITY /POSTHOC=TUKEY.

* 7. Test association between categorical variables.
CROSSTABS /TABLES=Region BY WouldRecommend /STATISTICS=CHISQ PHI /CELLS=COUNT ROW.

* 8. Model what predicts spend.
REGRESSION
  /DEPENDENT MonthlySpend_NGN
  /METHOD=ENTER MonthlyIncome_NGN000 PurchaseFrequency LoyaltyYears SatisfactionIndex.

* 9. Save the enriched, labeled file for future sessions.
SAVE OUTFILE='C:\UAC_Foods\UAC_Foods_Survey_Final.sav'.
```

---

## Chapter 16: Practice Exercises

Try writing the syntax yourself before checking the answer key.

1. Produce a frequency table of `Education`, sorted so the most common category appears first.
2. Create a new variable `SpendPerPurchase` equal to `MonthlySpend_NGN` divided by `PurchaseFrequency`.
3. Recode `LoyaltyYears` into a new variable `LoyaltyBand` with three categories: 0–2 years, 3–7 years, 8+ years.
4. Run a Chi-Square test of whether `Education` is associated with `WouldRecommend`.
5. Run an independent-samples t-test comparing `SatisfactionIndex` between respondents who would and wouldn't recommend the product.
6. Run a multiple regression predicting `OverallSatisfaction` from `Taste`, `Price`, `Packaging`, and `Availability`, and identify which predictor has the largest standardized coefficient (Beta).

### Answer Key

```spss
* 1.
FREQUENCIES VARIABLES=Education /ORDER=ANALYSIS /FORMAT=DFREQ.

* 2.
COMPUTE SpendPerPurchase = MonthlySpend_NGN / PurchaseFrequency.
VARIABLE LABELS SpendPerPurchase 'Average Spend per Purchase (Naira)'.
EXECUTE.

* 3.
RECODE LoyaltyYears (0 THRU 2=1)(3 THRU 7=2)(8 THRU HIGHEST=3) INTO LoyaltyBand.
VALUE LABELS LoyaltyBand 1 '0-2 years' 2 '3-7 years' 3 '8+ years'.
EXECUTE.

* 4.
CROSSTABS /TABLES=Education BY WouldRecommend /STATISTICS=CHISQ /CELLS=COUNT ROW.

* 5.
COMPUTE SatisfactionIndex = MEAN(Taste, Price, Packaging, Availability).
EXECUTE.
T-TEST GROUPS=WouldRecommend('Yes' 'No') /VARIABLES=SatisfactionIndex.

* 6.
REGRESSION
  /STATISTICS COEFF OUTS R ANOVA
  /DEPENDENT OverallSatisfaction
  /METHOD=ENTER Taste Price Packaging Availability.
* Compare the "Beta" (standardized coefficient) column in the Coefficients table across
* the four predictors; the largest absolute Beta is the strongest unique predictor.
```

---

## Chapter 17: Common Errors and How to Read Them

| Error message | What it usually means | Fix |
|---|---|---|
| "An unknown command was specified" | Usually a missing period on the *previous* command, so SPSS reads its next line as a new command | Add the missing `.` |
| "Variable ... not defined" | Misspelled variable name, or the file wasn't fully imported yet | Check `DISPLAY DICTIONARY.` output for exact spelling |
| "There is no dataset open" | Ran a command before `GET DATA`/`GET FILE` finished | Re-run the import first |
| Table looks empty / all missing | `MISSING` handling (LISTWISE vs. PAIRWISE) excluded all cases | Check which variables have missing data with `FREQUENCIES` |
| Numbers look like text (left-aligned) | Column was imported as a string when it should be numeric | Re-import with the correct `F8.0` format, or use `ALTER TYPE varname (F8.0).` |

---

## Where to Go From Here

Once comfortable with everything above, natural next steps include: `FACTOR` (factor analysis / PCA), `CLUSTER` and `QUICK CLUSTER` (segmentation), `LOGISTIC REGRESSION` (predicting a Yes/No outcome like WouldRecommend directly instead of using Chi-Square), and the `MIXED` / `GENLIN` commands for more advanced modeling. All follow the same syntax grammar you've learned here: `COMMAND /subcommand=value ... .`